# Cost-aware Stopping in SOEA

## Packages

In [ ]:
pip install cma==4.4.4 numpy pandas

In [ ]:
import os
from pathlib import Path
from importlib.metadata import version
import numpy as np
import pandas as pd
import cma
from cma import bbobbenchmarks as bn
from scipy.optimize import minimize

## Experiment

In [ ]:
# Settings
DIMS= [2,3,5]
FUNCTION_IDS=range(1,25)
INSTANCE_IDS=range(1,16)
CMA_CRITERIA=["tolfun","tolfunhist","tolflatfitness","tolstagnation"] #,"noeffectaxis"]
ONLINE_CRITERIA=CMA_CRITERIA+["cost_aware"]
ALL_CRITERIA=ONLINE_CRITERIA+["hindsight_best"]

SIGMA0=2.0
MAX_FE_FACTOR=10000
REF_INIT_RUNS=1000
RHO=10
T_MIN=50
M=5
OUTPUT_FILE=Path("cmaes_stopping_results.csv")
RESUME=True

if version("cma")!="4.4.4":
    raise RuntimeError(f"Please use cma==4.4.4. Current version: {version('cma')}")

# Reference initial optimality gap
def estimate_ref_gap(function_id,dim,n_runs=REF_INIT_RUNS):
    lam=4+int(3*np.log(dim))
    gaps=[]

    for r in range(n_runs):
        instance_id=INSTANCE_IDS[r%len(INSTANCE_IDS)]
        seed=10000+r
        rng=np.random.RandomState(seed)

        f=getattr(bn,f"F{function_id}")(instance_id)
        f_opt=float(f.getfopt())
        x0=rng.uniform(-4,4,size=dim)

        opts={
            "popsize":lam,
            "seed":seed,
            "verbose":-9,
            "signals_filename":""
        }
        es=cma.CMAEvolutionStrategy(x0,SIGMA0,opts)

        solutions=es.ask()
        values=np.asarray([f(x) for x in solutions],dtype=float)
        B0=float(np.min(values))

        gaps.append(B0-f_opt)

    return float(np.mean(gaps))


# Linear normalized loss and performance
def normalized_loss(best,f_opt,ref_gap):
    if ref_gap<=0:
        raise ValueError("Reference initial gap must be positive.")
    return float(max(best-f_opt,0.0)/ref_gap)

def performance_metric(loss,fes,max_fes,rho=RHO):
    return float(loss+rho*fes/max_fes)

# Weighted combination model
def unpack_theta(theta):
    return theta[0:3],theta[3:7],theta[7:11],theta[11:15],theta[15:19]

def combined_curve(t,theta):
    t=np.asarray(t,dtype=float)
    p1,p2,p3,p4,w=unpack_theta(theta)
    a1,A1,k1=p1
    a2,A2,c2,q2=p2
    a3,A3,k3,beta3=p3
    a4,A4,k4,q4=p4
    g1=a1+A1*np.exp(-k1*t)
    g2=a2+A2*(t+c2)**(-q2)
    g3=a3+A3*np.exp(-(k3*t)**beta3)
    g4=a4+A4/(1+k4*t)**q4
    return w[0]*g1+w[1]*g2+w[2]*g3+w[3]*g4

def initial_theta(y):
    y0,yend=float(y[0]),float(y[-1])
    asym=0.9*yend
    amp=max(y0-asym,1e-3)
    return np.array([
        asym,amp,0.05,
        asym,amp,1.0,1.0,
        asym,amp,0.05,1.0,
        asym,amp,0.05,1.0,
        0.25,0.25,0.25,0.25
    ])

PARAM_BOUNDS=[
    (0,1),(0,2),(1e-8,10),
    (0,1),(0,2),(1e-6,100),(0.05,10),
    (0,1),(0,2),(1e-8,10),(0.05,10),
    (0,1),(0,2),(1e-8,10),(0.05,10),
    (0,1),(0,1),(0,1),(0,1)
]

def fit_predict_next_loss(loss_history,previous_theta=None):
    try:
        y=np.asarray(loss_history,dtype=float)
        t=np.arange(len(y),dtype=float)
        t_next=float(len(y))
        current=float(y[-1])
        theta0=initial_theta(y) if previous_theta is None else previous_theta.copy()

        def objective(theta):
            pred=combined_curve(t,theta)
            return 1e12 if not np.all(np.isfinite(pred)) else float(np.mean((pred-y)**2))

        def next_pred(theta):
            p=float(combined_curve([t_next],theta)[0])
            return p if np.isfinite(p) else -1e12

        constraints=[
            {"type":"eq","fun":lambda th:np.sum(th[15:19])-1},
            {"type":"ineq","fun":lambda th:next_pred(th)},
            {"type":"ineq","fun":lambda th:current-next_pred(th)}   ## Elitist prediction condition 
        ]

        res=minimize(
            objective,theta0,method="SLSQP",
            bounds=PARAM_BOUNDS,
            constraints=constraints,
            options={"maxiter":1000,"ftol":1e-10,"disp":False}
        )

        if not res.success:
            return None,None,False

        pred=float(combined_curve([t_next],res.x)[0])
        valid=(
            np.isfinite(pred)
            and pred>=-1e-8
            and pred<=current+1e-8                ## Elitist prediction condition
            and abs(np.sum(res.x[15:19])-1)<=1e-6
        )
        return (pred,res.x,True) if valid else (None,None,False)

    except Exception:
        return None,None,False

# One CMA-ES/BBOB run
def run_one_problem(function_id,instance_id,dim,ref_gap):
    seed=instance_id
    rng=np.random.RandomState(seed)

    f=getattr(bn,f"F{function_id}")(instance_id)
    f_opt=float(f.getfopt())
    x0=rng.uniform(-4,4,size=dim)

    lam=4+int(3*np.log(dim))
    max_fes=MAX_FE_FACTOR*dim
    tstag=int(100+100*dim**1.5/lam)

    opts={
        "popsize":lam,"seed":seed,"verbose":-9,"signals_filename":"",
        "tolfun":1e-11,"tolfunhist":1e-12,
        "tolflatfitness":1,"tolstagnation":tstag
    }
    es=cma.CMAEvolutionStrategy(x0,SIGMA0,opts)

    best_so_far=np.inf
    loss_history=[]
    fe_history=[]
    best_history=[]
    perf_history=[]
    gen_history=[]

    first_trigger={}
    cost_counter=0
    previous_theta=None
    run_status="completed"
    error_message=""

    while es.countevals<max_fes:
        # CMA-ES trajectory
        try:
            solutions=es.ask()
            values=np.asarray([f(x) for x in solutions],dtype=float)

            # IMPORTANT: do not replace values near f_opt by f_opt
            candidate_best=float(np.min(values))
            es.tell(solutions,values)

            best_so_far=min(best_so_far,candidate_best)
            fes=int(es.countevals)
            generation=int(es.countiter)

            loss_t=normalized_loss(best_so_far,f_opt,ref_gap)
            perf_t=performance_metric(loss_t,fes,max_fes)

            loss_history.append(loss_t)
            fe_history.append(fes)
            best_history.append(best_so_far)
            perf_history.append(perf_t)
            gen_history.append(generation)

            # Five pycma stopping criteria
            stop_dict=dict(es.stop())
            for criterion in CMA_CRITERIA:
                if criterion in stop_dict and criterion not in first_trigger:
                    first_trigger[criterion]={
                        "fe":fes,"best":best_so_far,"loss":loss_t,
                        "performance":perf_t,"generation":generation
                    }

        except Exception as e:
            run_status="error"
            error_message=repr(e)
            break

        # Proposed cost-aware criterion
        if "cost_aware" not in first_trigger:
            if loss_t<=0:
                first_trigger["cost_aware"]={
                    "fe":fes,"best":best_so_far,"loss":loss_t,
                    "performance":perf_t,"generation":generation,
                    "predicted_next_loss":0.0,
                    "predicted_improvement":0.0,
                    "next_generation_cost":RHO*lam/max_fes
                }

            elif len(loss_history)>=T_MIN:
                pred,theta_hat,ok=fit_predict_next_loss(loss_history,previous_theta)

                if ok:
                    previous_theta=theta_hat
                    improvement=loss_t-pred
                    gen_cost=RHO*lam/max_fes
                    cost_counter=cost_counter+1 if improvement<=gen_cost else 0

                    if cost_counter>=M:
                        first_trigger["cost_aware"]={
                            "fe":fes,"best":best_so_far,"loss":loss_t,
                            "performance":perf_t,"generation":generation,
                            "predicted_next_loss":pred,
                            "predicted_improvement":improvement,
                            "next_generation_cost":gen_cost
                        }
                else:
                    cost_counter=0

    # Hindsight-best: earliest generation with minimum observed P_t
    if perf_history:
        hb_idx=int(np.argmin(perf_history))
        first_trigger["hindsight_best"]={
            "fe":fe_history[hb_idx],
            "best":best_history[hb_idx],
            "loss":loss_history[hb_idx],
            "performance":perf_history[hb_idx],
            "generation":gen_history[hb_idx]
        }

    final=None
    if perf_history:
        final={
            "fe":fe_history[-1],
            "best":best_history[-1],
            "loss":loss_history[-1],
            "performance":perf_history[-1],
            "generation":gen_history[-1]
        }

    def pack(criterion,r,triggered,reason,ctype):
        return {
            "dimension":dim,
            "function_id":function_id,
            "instance_id":instance_id,
            "seed":seed,
            "lambda":lam,
            "criterion":criterion,
            "criterion_type":ctype,
            "triggered":triggered,
            "stop_reason":reason,
            "stop_generation":np.nan if r is None else r["generation"],
            "stop_fe":np.nan if r is None else r["fe"],
            "best_so_far":np.nan if r is None else r["best"],
            "L_t":np.nan if r is None else r["loss"],
            "P_t":np.nan if r is None else r["performance"],
            "best_so_far_error":np.nan if r is None else r["best"]-f_opt,
            "f_opt":f_opt,
            "ref_gap":ref_gap,
            "ref_init_runs":REF_INIT_RUNS,
            "max_fe":max_fes,
            "rho":RHO,
            "T_min":T_MIN,
            "M":M,
            "predicted_next_loss":np.nan if r is None else r.get("predicted_next_loss",np.nan),
            "predicted_improvement":np.nan if r is None else r.get("predicted_improvement",np.nan),
            "next_generation_cost":np.nan if r is None else r.get("next_generation_cost",np.nan),
            "run_status":run_status,
            "error_message":error_message
        }

    rows=[]
    for criterion in ALL_CRITERIA:
        if criterion=="hindsight_best" and criterion in first_trigger:
            rows.append(pack(criterion,first_trigger[criterion],np.nan,"oracle_min_P","oracle"))
        elif criterion in first_trigger:
            rows.append(pack(criterion,first_trigger[criterion],True,"criterion","online"))
        elif run_status=="completed" and final is not None:
            rows.append(pack(criterion,final,False,"budget","online"))
        else:
            rows.append(pack(criterion,None,False,"error_before_trigger","online"))

    return rows

# Resume completed runs
def get_completed_runs():
    if not RESUME or not OUTPUT_FILE.exists():
        return set()

    df=pd.read_csv(OUTPUT_FILE)
    if df.empty:
        return set()

    counts=df.groupby(
        ["dimension","function_id","instance_id"]
    )["criterion"].nunique()

    return {
        tuple(idx) for idx,count in counts.items()
        if count==len(ALL_CRITERIA)
    }

# Full experiment
def run_experiment():
    OUTPUT_FILE.parent.mkdir(parents=True,exist_ok=True)

    if not RESUME and OUTPUT_FILE.exists():
        OUTPUT_FILE.unlink()

    # Estimate one reference initial gap for each function-dimension pair
    ref_gaps={}
    for dim in DIMS:
        for function_id in FUNCTION_IDS:
            ref_gap=estimate_ref_gap(function_id,dim)
            ref_gaps[(function_id,dim)]=ref_gap
            print(f"Reference gap: d={dim}, f={function_id}, G0_bar={ref_gap:.6g}")

    completed=get_completed_runs()
    total=len(DIMS)*len(FUNCTION_IDS)*len(INSTANCE_IDS)
    run_no=0

    for dim in DIMS:
        for function_id in FUNCTION_IDS:
            ref_gap=ref_gaps[(function_id,dim)]

            for instance_id in INSTANCE_IDS:
                run_no+=1
                key=(dim,function_id,instance_id)

                if key in completed:
                    print(f"[{run_no}/{total}] d={dim}, f={function_id}, i={instance_id}: completed")
                    continue

                print(f"[{run_no}/{total}] d={dim}, f={function_id}, i={instance_id}")

                rows=run_one_problem(
                    function_id,instance_id,dim,ref_gap
                )

                header=not OUTPUT_FILE.exists() or OUTPUT_FILE.stat().st_size==0
                pd.DataFrame(rows).to_csv(
                    OUTPUT_FILE,mode="a",header=header,index=False
                )

    print(f"\nCompleted. Results saved to {OUTPUT_FILE}")

if __name__=="__main__":
    run_experiment()